# Assemble subset .h5ad files

For QC and inspection of our cell type labels, we'll assemble samples based on subject metadata to get some reusable units that will be manageable for downstream analysis. 

To each of these subsets, we'll add our cell type label predictions and scrublet scores and calls.

We'll carry these subsets forward into QC filtering and cell type-based doublet and mislabeling clean-up in later notebooks.

## Load Packages

`anndata`: Data structures for scRNA-seq  
`datetime`: date and time functions  
`h5py`: HDF5 file I/O  
`hisepy`: The HISE SDK for Python  
`os`: operating system calls  
`pandas`: DataFrame data structures  
`re`: Regular expressions  
`scanpy`: scRNA-seq analysis  
`scipy.sparse`: Spare matrix data structures  

In [1]:
import anndata
from datetime import date
import h5py
import hisepy
import os
import pandas as pd
from pandas.api.types import is_object_dtype
import re
import scanpy as sc
import scipy.sparse as scs
import numpy as np

## Helper functions

In [2]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [3]:
hisepy.__version__

'0.3.0'

In [4]:
def read_csv_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_csv(cache_file)
    return res

In [5]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

Functions to read pipeline .h5 files as anndata

In [6]:
# define a function to read count data
def read_mat(h5_con):
    mat = scs.csc_matrix(
        (h5_con['matrix']['data'][:], # Count values
         h5_con['matrix']['indices'][:], # Row indices
         h5_con['matrix']['indptr'][:]), # Pointers for column positions
        shape = tuple(h5_con['matrix']['shape'][:]) # Matrix dimensions
    )
    return mat

# define a function to read obeservation metadata (i.e. cell metadata)
def read_obs(h5con):
    bc = h5con['matrix']['barcodes'][:]
    bc = [x.decode('UTF-8') for x in bc]

    # Initialized the DataFrame with cell barcodes
    obs_df = pd.DataFrame({ 'barcodes' : bc })

    # Get the list of available metadata columns
    obs_columns = h5con['matrix']['observations'].keys()
    
    # For each column
    for col in obs_columns:
        # Read the values
        values = h5con['matrix']['observations'][col][:]
        # Check for byte storage
        if(isinstance(values[0], (bytes, bytearray))):
            # Decode byte strings
            values = [x.decode('UTF-8') for x in values]
        # Add column to the DataFrame
        obs_df[col] = values

    obs_df = obs_df.set_index('barcodes', drop = False)
    
    return obs_df

# define a function to construct anndata object from a h5 file
def read_h5_anndata(h5_file):
    h5_con = h5py.File(h5_file, mode = 'r')
    # extract the expression matrix
    mat = read_mat(h5_con)
    # extract gene names
    genes = h5_con['matrix']['features']['name'][:]
    genes = [x.decode('UTF-8') for x in genes]
    # extract metadata
    obs_df = read_obs(h5_con)
    # construct anndata
    adata = anndata.AnnData(mat.T,
                             obs = obs_df)
    # make sure the gene names aligned
    adata.var_names = genes

    adata.var_names_make_unique()
    return adata

In [7]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Read sample metadata from HISE

In [9]:
in_uuids = []

In [10]:
## sample_meta_uuid from previous step
sample_meta_uuid = 'cb01d5b1-40d8-4940-b61c-86d78de4528b'
sample_meta = read_csv_uuid(sample_meta_uuid)
in_uuids.append(sample_meta_uuid)

We only need to keep some of the metadata columns that pertain to cohort, subject, and sample. We'll also keep the originating File GUID to help us keep track of provenance. Let's select just these columns:

In [17]:
sample_meta.columns.tolist()

['Unnamed: 0',
 'lastUpdated',
 'sample.id',
 'sample.bridgingControl',
 'sample.sampleKitGuid',
 'sample.visitName',
 'sample.visitDetails',
 'sample.drawDate',
 'sample.daysSinceFirstVisit',
 'file.id',
 'file.name',
 'file.batchID',
 'file.panel',
 'file.pool',
 'file.fileType',
 'file.userTags.details',
 'file.userTags.group',
 'file.userTags.name',
 'file.userTags.origin',
 'file.userTags.other',
 'file.userTags.version',
 'file.majorVersion',
 'subject.id',
 'subject.biologicalSex',
 'subject.birthYear',
 'subject.ethnicity',
 'subject.partnerCode',
 'subject.race',
 'subject.subjectGuid',
 'cohort.cohortGuid',
 'specimens.specimenType',
 'specimens.specimenGuid']

In [18]:
set(sample_meta[ 'file.batchID'])

{'B053',
 'B060',
 'B063',
 'B093',
 'B094',
 'B096',
 'B102',
 'B128',
 'B130',
 'B132',
 'B138',
 'B144',
 'B148',
 'B161'}

In [19]:
keep_meta = [
    'cohort.cohortGuid',
    'subject.subjectGuid', 'subject.biologicalSex', 
    'subject.race', 'subject.ethnicity', 'subject.birthYear',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawDate',
    'specimens.specimenGuid','specimens.specimenType',
    'file.id', 'file.batchID'
]

In [20]:
sample_meta = sample_meta[keep_meta]

## Read labels and doublet calls from HISE

In [21]:
## search_id values from previous steps
label_search_id = 'silicon-californium-cerium'
doublet_search_id = 'magnesium-osmium-arsenic'

search_string = '|'.join([label_search_id, doublet_search_id])

Retrieve files stored in our HISE project store

In [22]:
ps_df = hisepy.list_files_in_project_store('Dyna_IHandA')
ps_df = ps_df[['id', 'name']]

In [25]:
ps_df

,id,name
0,0180d34e-bcb4-4b81-9214-818641ec54e3,AIFI-2024-02-05T23:47:28.925924866Z/br1-br2_cl...
1,223b4aa9-19fc-41e1-8bea-43682e5ac278,AIFI-2024-02-08T21:29:39.75691749Z/ref_h5_meta...
2,dca28828-3485-420e-872e-fee33d1898cb,AIFI-2024-02-09T23:39:41.138930843Z/EXP-00776_...
3,c945e0c5-4bfd-4b18-a990-d496252d5de7,AIFI-2024-02-10T06:30:21.185363381Z/ref_subjec...
4,cf3a852c-3222-4004-9144-30d43680ef2b,AIFI-2024-02-10T06:30:28.260682402Z/ref_subjec...
5,7184a685-7b0a-4edc-8730-f821277d9375,AIFI-PSP-2024-02-13T00:15:54.372169127Z/EXP-00...
6,d3ad68f3-b43c-4c6f-992c-8877b14d08f6,AIFI-2024-03-14T19:57:18.196621831Z/diha_AIFI_...
7,7df0f24c-a764-4473-bcb5-180022e85a4c,AIFI-2024-03-14T19:58:19.322053234Z/diha_AIFI_...
8,0b126dd5-7a35-4f86-85a8-6af37a4e1842,AIFI-2024-03-14T22:33:45.917841335Z/diha_AIFI_...
9,546d0304-714c-4244-8899-a665283eec2e,AIFI-2024-03-14T22:33:51.869481839Z/diha_AIFI_...


Filter for files from the previous notebook using our search_id

In [26]:
search_df = ps_df[ps_df['name'].str.contains(search_string)]
search_df = search_df[search_df['name'].str.contains('.parquet')]
search_df = search_df.sort_values('name')

In [27]:
search_df['name'].tolist()

['magnesium-osmium-arsenic/up1_scrublet_results_2024-08-16.parquet',
 'silicon-californium-cerium/ra_celltypist_AIFI_L1_2024-08-15.parquet',
 'silicon-californium-cerium/ra_celltypist_AIFI_L2_2024-08-15.parquet',
 'silicon-californium-cerium/ra_celltypist_AIFI_L3_2024-08-15.parquet']

In [28]:
search_df

,id,name
55,e7c9a970-5075-4af5-a797-403b03848019,magnesium-osmium-arsenic/up1_scrublet_results_...
49,041ef46f-483d-4a3d-900c-64c97072da7c,silicon-californium-cerium/ra_celltypist_AIFI_...
51,a5300b3e-4739-47be-8fc0-cb1039ff710d,silicon-californium-cerium/ra_celltypist_AIFI_...
53,5942d3fa-9efe-43e2-9e48-81b6cecdf089,silicon-californium-cerium/ra_celltypist_AIFI_...


## Read and combine label sets to simplify merges

In [29]:
label_list = []
for uuid in search_df['id']:
    res = read_parquet_uuid(uuid)
    res = res.set_index('barcodes', drop = True)
    label_list.append(res)

In [30]:
all_labels = pd.concat(label_list, axis = 1)

In [31]:
all_labels = all_labels.reset_index(drop = False)
all_labels.head()

,barcodes,predicted_doublet,doublet_score,AIFI_L1,over_clustering,majority_voting,AIFI_L1_score,AIFI_L2,over_clustering,majority_voting,AIFI_L2_score,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,b9993ec4872b11ebb599de5a8c2059ae,False,0.060170,T cell,70,T cell,0.997307,Proliferating NK cell,70,Memory CD4 T cell,0.027022,Memory CD4 Treg,70,CM CD4 T cell,0.999993
1,b99942fc872b11ebb599de5a8c2059ae,False,0.013071,T cell,22,T cell,0.999338,Memory CD8 T cell,22,Memory CD8 T cell,0.955047,KLRF1- GZMB+ CD27- EM CD8 T cell,22,KLRF1+ GZMB+ CD27- EM CD8 T cell,1.000000
2,b9996f66872b11ebb599de5a8c2059ae,False,0.120700,T cell,142,T cell,1.000000,Naive CD8 T cell,142,Naive CD8 T cell,0.984279,Core naive CD8 T cell,142,Core naive CD8 T cell,1.000000
3,b99971c8872b11ebb599de5a8c2059ae,False,0.018831,Monocyte,23,Monocyte,0.999258,CD14 monocyte,23,CD14 monocyte,0.997393,Core CD14 monocyte,23,Core CD14 monocyte,1.000000
4,b9997290872b11ebb599de5a8c2059ae,False,0.026079,Monocyte,46,Monocyte,0.999999,CD14 monocyte,46,CD14 monocyte,0.999497,ISG+ CD14 monocyte,46,Core CD14 monocyte,1.000000


## Define subsets

In [33]:
len(sample_meta)

218

In [36]:
subset_column = 'subset_grp'

grps = ["set1", "set2"]

num_repeats = len(sample_meta) // len(grps)


assigned_values = np.repeat(grps, num_repeats)

np.random.seed(202405)
np.random.shuffle(assigned_values)

sample_meta.loc[:, subset_column] = assigned_values

In [37]:
subset_counts = sample_meta[subset_column].value_counts()
subset_counts

subset_grp
set1    109
set2    109
Name: count, dtype: int64

In [38]:
subset_sample_meta = sample_meta.groupby(subset_column)

In [39]:
sample_meta.head()

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.race,subject.ethnicity,subject.birthYear,sample.sampleKitGuid,sample.visitName,sample.drawDate,specimens.specimenGuid,specimens.specimenType,file.id,file.batchID,subset_grp
0,UP1,UP1001,Female,African American,Non-Hispanic origin,2009,KT00826,Flu Year 1 Day 90,2020-12-01T00:00:00Z,PB00826-07,PBMC,4aa6a6ac-e14c-4c41-9c44-9783c4252f20,B053,set1
1,UP1,UP1001,Female,African American,Non-Hispanic origin,2009,KT00891,Flu Year 1 Day 180-360,2021-03-01T00:00:00Z,PL00891-20,Plasma,c70d0dfe-4223-467f-998b-254247765793,B063,set2
2,UP1,UP1001,Female,African American,Non-Hispanic origin,2009,KT00124,Flu Year 1 Pre-Vac 7-12 Weeks,2020-08-01T00:00:00Z,TP00124-01,Tempus Tube,8eb7283b-0394-4ca9-8646-15ae4a206fe6,B053,set1
3,UP1,UP1001,Female,African American,Non-Hispanic origin,2009,KT00199,Flu Year 1 Day 7,2020-09-01T00:00:00Z,PL00199-23,NaN,92945be5-51fc-4aea-9bcc-fde6bacd2301,B053,set1
4,UP1,UP1001,Female,African American,Non-Hispanic origin,2009,KT00943,COVID-19 Visit 3,2021-06-01T00:00:00Z,PB00943-11,PBMC,6db37504-2857-4774-8263-2331408cc405,B093,set2


## Read and assemble anndata objects for each subset

In [42]:
df.shape

(109, 14)

In [43]:
group_input_dict = {}
group_adata_dict = {}
for group, df in subset_sample_meta:
    group_name = group
    #df = df.iloc[0:5] # Remove for full run
    group_uuids = df['file.id'].tolist()
    group_input_dict[group_name] = group_uuids

    # Cache files
    cache_dir = '/home/jupyter/cache'
    cache_paths = []
    for uuid in group_uuids:
        cache_path = '{d}/{u}'.format(d = cache_dir, u = uuid)
        if not os.path.isdir(cache_path):
            hise_res = hisepy.reader.cache_files([uuid])
        cache_paths.append(cache_path)

    # Get cached file paths
    cache_files = []
    for cache_path in cache_paths:
        fn = os.listdir(cache_path)[0]
        cache_files.append('{d}/{f}'.format(d = cache_path, f = fn))

    # Read cached files as anndata
    adata_list = []
    for cache_file in cache_files:
        adata = read_h5_anndata(cache_file)
        adata_list.append(adata)
    group_adata = sc.concat(adata_list)
    
    group_adata_dict[group_name] = group_adata

downloading fileID: cf8a34e7-fab9-46ef-bfca-06d76dce5005
Files have been successfully downloaded!
downloading fileID: d817713c-74b0-4fa1-bde5-d4d9a35ae4a8
Files have been successfully downloaded!
downloading fileID: f2f12b37-7adc-45c7-b5fc-73cedcb44c1e
Files have been successfully downloaded!
downloading fileID: f68c79f5-8bea-48d6-b6b2-728e7c7dd8d2
Files have been successfully downloaded!
downloading fileID: 8c81c819-4e8b-45b2-8926-9b388e4f9049
Files have been successfully downloaded!
downloading fileID: 86dd368f-1ec3-46d9-83a5-a76272e4c784
Files have been successfully downloaded!
downloading fileID: 9bef6cc0-5535-4fe0-b6b3-9de0a68c28ef
Files have been successfully downloaded!
downloading fileID: 0335062a-299b-406c-a939-3eddd83b2c2e
Files have been successfully downloaded!
downloading fileID: 79e9c5f6-c9d8-4f57-95ba-1786abc1d4aa
Files have been successfully downloaded!
downloading fileID: 84412166-dd18-45d9-b407-23ba4ac1a77f
Files have been successfully downloaded!
downloading fileID: 

In [44]:
group_adata

AnnData object with n_obs × n_vars = 1981919 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id'

In [45]:
group_adata_dict.keys()

dict_keys(['set1', 'set2'])

In [46]:
for group_name, adata in group_adata_dict.items():
    print('{g}: {n} cells'.format(g = group_name, n = adata.shape[0]))

set1: 1929009 cells
set2: 1981919 cells


In [47]:
sum(adata.shape[0] for adata in group_adata_dict.values())

3910928

## Update Observations with additional metadata

Now, we'll add the sample metadata, CMV status, and BMI data to our scRNA-seq data.

First, we'll convert `pbmc_sample_id` to `sample.sampleKitGuid` using a regular expression. PBMC samples are derived from kits in our LIMS system, so both share the same numerical core. The difference is that there can be multiple PBMC samples collected at the same time, so PBMC samples are prefixed with PB to indicate their sample type, and suffixed with -XX to indicate an aliquot number.

In [48]:
def sample_to_kit(sample):
    kit = re.sub('PB([0-9]+)-.+','KT\\1',sample)
    return(kit)

To keep things tidy, we'll also drop the `seurat_pbmc_type`, `seurat_pbmc_type_score`, and UMAP coordinates generated by our processing pipeline. These cell type assignments are from a now-outdated reference dataset, and the UMAP coordinates are generated for viewing individual samples - not helpful for our full dataset.

In [49]:
drop_columns = [
    'seurat_pbmc_type','seurat_pbmc_type_score',
    'umap_1', 'umap_2'
]

Then, we'll add our new sample data with a left join on the `sample.sampleKitGuid` values, and add our labels and doublet calls with a left join on cell `barcodes`.

Next, we'll convert all of our text columns to categorical. This is used to make storage of text data more efficient when we write our output file, as we'll only need to store a single instance of our strings.

We do this for all columns except barcodes, which we need to retain as a string type for use as an index.

Finally, we'll add these back to our anndata object

In [50]:
for group_name, adata in group_adata_dict.items():
    print('{g}: {p}'.format(g = group_name, p = adata.obs['pbmc_sample_id'].str.startswith('PB').all()))

set1: True
set2: True


In [51]:
for group_name, adata in group_adata_dict.items():
    print(group_name)
    obs = adata.obs
    
    # Convert sample.sampleKitGuid
    print('converting ids')
    obs['sample.sampleKitGuid'] = [sample_to_kit(sample) for sample in obs['pbmc_sample_id']]
    
    # Drop old columns
    obs = obs.drop(drop_columns, axis = 1)
    
    print('merging sample metadata')
    # Add new metadata
    obs = obs.merge(
        sample_meta,
        how = 'left',
        on = 'sample.sampleKitGuid'
    )
    
    print('merging labels')
    # Add labels
    obs = obs.merge(all_labels, how = 'left', on = 'barcodes')
    
    print('converting to categorical')
    # Convert to categorical
    cat_obs = obs
    for i in range(cat_obs.shape[1]):
        col_name = cat_obs.dtypes.index.tolist()[i]
        col_type = cat_obs.dtypes[col_name]
        if col_name == 'barcodes':
            cat_obs[col_name] = cat_obs[col_name].astype(str)
        elif is_object_dtype(col_type):
            cat_obs[col_name] = cat_obs[col_name].astype('category')
    cat_obs = cat_obs.set_index('barcodes', drop = False)
    
    # Assign final observations back to anndata
    adata.obs = cat_obs
    
    group_adata_dict[group_name] = adata

set1
converting ids
merging sample metadata
merging labels
converting to categorical
set2
converting ids
merging sample metadata
merging labels
converting to categorical


## Write assembled data to disk

In [52]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [53]:
out_h5ads = {}
for group_name, adata in group_adata_dict.items():
    out_h5ad = 'output/up1_pbmc_{g}_raw_labeled_{d}.h5ad'.format(g = group_name, d = date.today())
    adata.write_h5ad(out_h5ad)
    out_h5ads[group_name] = out_h5ad

In [54]:
out_csvs = {}
out_parquets = {}
for group_name, adata in group_adata_dict.items():
    obs = adata.obs
    
    out_csv = 'output/up1_pbmc_{g}_raw_labeled_meta_{d}.csv'.format(g = group_name, d = date.today())
    obs.to_csv(out_csv)
    out_csvs[group_name] = out_csv

    out_parquet = 'output/up1_pbmc_{g}_raw_labeled_meta_{d}.parquet'.format(g = group_name, d = date.today())

    obs = obs.loc[:, ~obs.columns.duplicated()] ## remove duplicate columns 
    obs.to_parquet(out_parquet)
    out_parquets[group_name] = out_parquet

## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [55]:
ss = hisepy.get_study_spaces()

In [56]:
ss

[{'id': '0b6bf907-6985-40e0-944d-677ac932677f',
  'accountGuid': 'b34b9c66-aa5d-45d0-9569-f78ca8f6813f',
  'projectGuid': '7ef501f4-79ea-45f0-ad97-6ed237cd5dd1',
  'driveId': '0AEgFtzSiOc7hUk9PVA',
  'auditInfo': {'added': '2024-08-14T20:42:25.668Z',
   'addedUser': 'temi.adewunmi@alleninstitute.org',
   'lastModified': '2024-08-19T17:19:23.424Z',
   'lastModifiedBy': 'temi.adewunmi@alleninstitute.org',
   'version': 1},
  'name': 'UP1 scRNAseq Study',
  'description': 'Project store for the work done on the scRNAseq samples from the UP1 cohort',
  'externalDrive': {'name': 'up1-studyExternal',
   'id': '1-wdxDzX3eTR2jYmdPEjJXE2HWtujRNWD',
   'parents': ['1NCAXL8ZmeQJawC2wRfG6si-O0CiMW3dN'],
   'webContentLink': '',
   'webViewLink': 'https://drive.google.com/drive/folders/1-wdxDzX3eTR2jYmdPEjJXE2HWtujRNWD',
   'iconLink': 'https://drive-thirdparty.googleusercontent.com/16/type/application/vnd.google-apps.folder+48+shared'},
  'reportDrive': {'name': 'up1-studyReport',
   'id': '1G9tr2

In [57]:

print(ss[0]['name'])
print(ss[0]['id'])
study_space_uuid = ss[0]['id']

title = 'Labeled Raw scRNA-seq Assembly {d}'.format(d = date.today())

UP1 scRNAseq Study
0b6bf907-6985-40e0-944d-677ac932677f


In [58]:
search_id = element_id()
search_id

'manganese-fermium-selenium'

In [59]:
in_files = [sample_meta_uuid] + search_df['id'].tolist()
in_files = in_files + sample_meta['file.id'].tolist()

In [60]:
len(in_files)

223

In [61]:
in_files[0:10]

['cb01d5b1-40d8-4940-b61c-86d78de4528b',
 'e7c9a970-5075-4af5-a797-403b03848019',
 '041ef46f-483d-4a3d-900c-64c97072da7c',
 'a5300b3e-4739-47be-8fc0-cb1039ff710d',
 '5942d3fa-9efe-43e2-9e48-81b6cecdf089',
 '4aa6a6ac-e14c-4c41-9c44-9783c4252f20',
 'c70d0dfe-4223-467f-998b-254247765793',
 '8eb7283b-0394-4ca9-8646-15ae4a206fe6',
 '92945be5-51fc-4aea-9bcc-fde6bacd2301',
 '6db37504-2857-4774-8263-2331408cc405']

In [62]:
out_files = list(out_h5ads.values()) + list(out_csvs.values()) + list(out_parquets.values())

In [63]:
out_files

['output/up1_pbmc_set1_raw_labeled_2024-08-19.h5ad',
 'output/up1_pbmc_set2_raw_labeled_2024-08-19.h5ad',
 'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.csv',
 'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.csv',
 'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.parquet',
 'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.parquet']

In [64]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/up1_pbmc_set1_raw_labeled_2024-08-19.h5ad', 'output/up1_pbmc_set2_raw_labeled_2024-08-19.h5ad', 'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.csv', 'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.csv', 'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.parquet', 'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.parquet']. Do you truly want to proceed?


(y/n) y


{'trace_id': '51b4f6c4-5ea3-41bc-b97d-0abe4bdce01d',
 'files': ['output/up1_pbmc_set1_raw_labeled_2024-08-19.h5ad',
  'output/up1_pbmc_set2_raw_labeled_2024-08-19.h5ad',
  'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.csv',
  'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.csv',
  'output/up1_pbmc_set1_raw_labeled_meta_2024-08-19.parquet',
  'output/up1_pbmc_set2_raw_labeled_meta_2024-08-19.parquet']}

In [65]:
import session_info
session_info.show()

#### Notes
1. removing majority voting and over clustering columns, rename these 2 columns based on levels?

In [66]:
all_labels.head()

,barcodes,predicted_doublet,doublet_score,AIFI_L1,over_clustering,majority_voting,AIFI_L1_score,AIFI_L2,over_clustering,majority_voting,AIFI_L2_score,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,b9993ec4872b11ebb599de5a8c2059ae,False,0.060170,T cell,70,T cell,0.997307,Proliferating NK cell,70,Memory CD4 T cell,0.027022,Memory CD4 Treg,70,CM CD4 T cell,0.999993
1,b99942fc872b11ebb599de5a8c2059ae,False,0.013071,T cell,22,T cell,0.999338,Memory CD8 T cell,22,Memory CD8 T cell,0.955047,KLRF1- GZMB+ CD27- EM CD8 T cell,22,KLRF1+ GZMB+ CD27- EM CD8 T cell,1.000000
2,b9996f66872b11ebb599de5a8c2059ae,False,0.120700,T cell,142,T cell,1.000000,Naive CD8 T cell,142,Naive CD8 T cell,0.984279,Core naive CD8 T cell,142,Core naive CD8 T cell,1.000000
3,b99971c8872b11ebb599de5a8c2059ae,False,0.018831,Monocyte,23,Monocyte,0.999258,CD14 monocyte,23,CD14 monocyte,0.997393,Core CD14 monocyte,23,Core CD14 monocyte,1.000000
4,b9997290872b11ebb599de5a8c2059ae,False,0.026079,Monocyte,46,Monocyte,0.999999,CD14 monocyte,46,CD14 monocyte,0.999497,ISG+ CD14 monocyte,46,Core CD14 monocyte,1.000000


In [67]:
all_labels.shape

(3910928, 15)